# Cadrille — Google Colab

Supports:
- **Eval**: run inference + metrics on a checkpoint (free T4 / Pro A100)
- **SFT smoke test**: 50-step training run to verify the pipeline
- **RL fine-tuning**: Dr. CPPO on H100 (Pro+ recommended)
- **Monitoring**: sync W&B offline runs

> Mount Google Drive first — data and checkpoints survive session restarts there.

In [ ]:
# ── [1] Mount Drive ──────────────────────────────────────────────────────────
# All data and checkpoints live here — they survive session restarts.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT  = '/content/drive/MyDrive/cadrille'   # adjust if needed
DRIVE_DATA  = f'{DRIVE_ROOT}/data'
DRIVE_CKPTS = f'{DRIVE_ROOT}/checkpoints'

import os
os.makedirs(DRIVE_DATA,  exist_ok=True)
os.makedirs(DRIVE_CKPTS, exist_ok=True)
print('Drive mounted. Paths:', DRIVE_DATA, DRIVE_CKPTS)

In [ ]:
# ── [2] Clone repo ───────────────────────────────────────────────────────────
import os

REPO = '/content/cadrille'
if not os.path.exists(REPO):
    !git clone https://github.com/miachen0401/cadrille.git {REPO}
else:
    !git -C {REPO} pull   # update to latest

%cd {REPO}

In [ ]:
# ── [3] Install dependencies ─────────────────────────────────────────────────
# flash-attn: prebuilt wheel when available (~30 s); MAX_JOBS=8 source build as fallback.
# cadquery:   cadquery-ocp MUST be installed before cadquery to avoid setup.py egg_info errors.

import sys, subprocess, os
import torch

# ── flash-attn ────────────────────────────────────────────────────────────────
# Build the exact wheel filename from the runtime CUDA / torch / Python versions.
# e.g. flash_attn-2.7.2.post1+cu12torch2.5-cp311-cp311-linux_x86_64.whl
_cuda = torch.version.cuda.split('.')[0]            # "12" from "12.4"
_tmm  = '.'.join(torch.__version__.split('.')[:2])  # "2.5" from "2.5.1"
_py   = f"cp{sys.version_info.major}{sys.version_info.minor}"  # "cp311"
_whl  = f"flash_attn-2.7.2.post1+cu{_cuda}torch{_tmm}-{_py}-{_py}-linux_x86_64.whl"
_url  = (f"https://github.com/Dao-AILab/flash-attention/releases/"
         f"download/v2.7.2.post1/{_whl}")

print(f"flash-attn: trying prebuilt wheel {_whl} ...")
_r = subprocess.run(["pip", "install", "-q", _url], capture_output=True, text=True)
if _r.returncode == 0:
    print("  → installed from prebuilt wheel (~30 s) ✓")
else:
    print("  → wheel not found; building from source with MAX_JOBS=8 (~10 min) ...")
    subprocess.run(
        ["pip", "install", "-q", "--no-build-isolation", "flash-attn==2.7.2.post1"],
        env={**os.environ, "MAX_JOBS": "8"}, check=True)
    print("  → built from source ✓")

# ── cadquery ──────────────────────────────────────────────────────────────────
# Install OCP bindings FIRST, then cadquery — reversing this order (or doing it
# in one pip call) causes "python setup.py egg_info did not run successfully".
# cadquery-ocp 7.7.2 has wheels for Python 3.9–3.11; 7.7.2.1 adds Python 3.12.
_ocp = "7.7.2.1" if sys.version_info >= (3, 12) else "7.7.2"
subprocess.run(["pip", "install", "-q", f"cadquery-ocp=={_ocp}"], check=True)
subprocess.run(["pip", "install", "-q", "cadquery==2.4.0"], check=True)
print(f"cadquery 2.4.0 + cadquery-ocp {_ocp} ✓")

# ── everything else ───────────────────────────────────────────────────────────
subprocess.run(["pip", "install", "-q",
    "transformers==4.50.3",
    "accelerate==0.34.2",
    "qwen-vl-utils==0.0.10",
    "bitsandbytes>=0.43.0",
    "trimesh==4.5.3",
    "open3d",
    "scipy==1.14.1",
    "wandb",
    "tqdm",
    "pyyaml",
], check=True)

# ── verify ────────────────────────────────────────────────────────────────────
p = torch.cuda.get_device_properties(0)
print(f"\nGPU: {p.name}  {p.total_memory // 1024**3} GB  |  CUDA {torch.version.cuda}  |  Python {sys.version.split()[0]}")

In [ ]:
# ── [4] Link data & checkpoints from Drive ───────────────────────────────────
import os

if not os.path.exists('data'):
    os.symlink(DRIVE_DATA, 'data')

if not os.path.exists('checkpoints'):
    os.symlink(DRIVE_CKPTS, 'checkpoints')

print('data/      →', os.listdir('data') or '(empty)')
print('checkpoints/ →', os.listdir('checkpoints') or '(empty)')

In [ ]:
# ── [5] Download training & validation data (run once, persists on Drive) ──────
# Downloads DeepCAD test meshes (RL training + validation) and Fusion360 test
# meshes (validation only). Each is a separate HuggingFace dataset repo.
# Skip this cell if both folders already exist in data/.

import os

if not os.path.exists('data/deepcad_test_mesh'):
    # ~1 GB — DeepCAD test split (STL meshes); used as RL training prompts + eval
    !huggingface-cli download maksimko123/deepcad_test_mesh \
        --repo-type dataset \
        --local-dir data/deepcad_test_mesh

if not os.path.exists('data/fusion360_test_mesh'):
    # ~200 MB — Fusion360 test split; used for cross-dataset validation only
    !huggingface-cli download maksimko123/fusion360_test_mesh \
        --repo-type dataset \
        --local-dir data/fusion360_test_mesh

print('Data ready:')
!ls data/deepcad_test_mesh | head -5
!echo "  ... $(ls data/deepcad_test_mesh | wc -l) deepcad meshes"
!echo "  ... $(ls data/fusion360_test_mesh | wc -l) fusion360 meshes"

In [ ]:
# ── [6] Download SFT checkpoint (run once, persists on Drive) ─────────────────
# Starting point for RL fine-tuning.
# Skip if checkpoints/cadrille-sft already exists on Drive.

import os

SFT_CKPT = 'checkpoints/cadrille-sft'
if not os.path.exists(SFT_CKPT):
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id='maksimko123/cadrille',
        local_dir=SFT_CKPT)
    print(f'Downloaded → {SFT_CKPT}')
else:
    print(f'Already exists: {SFT_CKPT}')

In [ ]:
# ── [7] W&B login ────────────────────────────────────────────────────────────
import wandb
wandb.login()   # paste your API key from wandb.ai/authorize

In [ ]:
# ── [8] RL fine-tuning on H100 ───────────────────────────────────────────────
# Uses configs/rl/h100.yaml:
#   G=16, top_N=4, batch_updates=3, max_new_tokens=400
#   Validation: 100 DeepCAD + 100 Fusion360, pc + img modalities
#
# Checkpoints saved to Drive every 500 steps; eval every 500 steps.
# W&B run ID is saved to checkpoints/<run_name>/wandb_run_id.txt — the resume
# cell [9] will automatically attach to the same W&B run.
#
# Expected throughput on H100: ~8-12 steps/min → 50k steps in ~3-4 days.

RUN_NAME = 'cadrille-rl-h100-v1'   # change for a new run; keep same to resume

!python rl/train.py \
    --config  configs/rl/h100.yaml \
    --run-name {RUN_NAME}

In [ ]:
# ── [9] Resume after session timeout ─────────────────────────────────────────
# Colab sessions timeout (~12h Pro, ~24h Pro+).
# When your session restarts:
#   1. Re-run cells [1] – [7] (Drive mount, clone, install, link, login)
#   2. Run this cell to continue from the latest saved checkpoint.
#
# The step counter resumes at the checkpoint step (no duplicate steps logged).
# W&B attaches to the original run via the saved run ID in wandb_run_id.txt.

import glob

RUN_NAME = 'cadrille-rl-h100-v1'   # must match the original run

ckpt_dirs = sorted(glob.glob(f'checkpoints/{RUN_NAME}/checkpoint-[0-9]*'),
                   key=lambda p: int(p.rsplit('-', 1)[-1]))
if not ckpt_dirs:
    print('No checkpoints found — start a fresh run with cell [8] instead.')
else:
    latest = ckpt_dirs[-1]
    step   = int(latest.rsplit('-', 1)[-1])
    print(f'Resuming from: {latest}  (step {step} → {50000})')
    !python rl/train.py \
        --config           configs/rl/h100.yaml \
        --run-name         {RUN_NAME} \
        --checkpoint-path  {latest}